# Full-match A100 test
Test input: JaHbat vs FC-1, September 13, 2026 (Drive file supplied by the user, 28.3 GB).
Select the Colab A100 kernel in VS Code, then run all cells. The notebook verifies the GPU, installs the pinned model, downloads the exact input, and processes the entire video without cached scores. Accuracy requires ground-truth annotations. Outputs remain in the Colab runtime under `outputs/full_match_a100`; download them before disconnecting.


In [ ]:
import torch, subprocess, sys, json, time
from pathlib import Path
assert torch.cuda.is_available(), "Connect the Colab A100 kernel"
GPU = torch.cuda.get_device_name(0)
assert "A100" in GPU, f"Expected A100, got {GPU}"
print("Verified GPU:", GPU)
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], text=True))


In [ ]:
REPO = Path("/content/TEED-match-tagger")
REVISION = "ed3b0943e43646591b32407c0ad23a1951921834"
def run(*args):
    subprocess.run(list(args), check=True)
if not REPO.exists():
    run("git", "clone", "https://github.com/jmoussa99/TEED-match-tagger.git", str(REPO))
assert subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip() == "", "Remote checkout has changes; inspect before setup"
run("git", "-C", str(REPO), "checkout", REVISION)
MODEL = REPO / "private/T-DEED"
if not MODEL.exists():
    run("git", "clone", "https://github.com/arturxe2/T-DEED.git", str(MODEL))
run("git", "-C", str(MODEL), "checkout", "d953b4de8a2ad941c565e1683c5b07255cbe644e")
patch = str(REPO / "app/patches/tdeed_mps.patch")
if subprocess.run(["git", "-C", str(MODEL), "apply", "--check", patch], capture_output=True).returncode == 0:
    run("git", "-C", str(MODEL), "apply", patch)
else:
    run("git", "-C", str(MODEL), "apply", "--reverse", "--check", patch)
run(sys.executable, "-m", "pip", "install", "timm", "gdown", "pandas", "opencv-python-headless")
CKPT = MODEL / "checkpoints/SoccerNetBall/SoccerNetBall_challenge2/checkpoint_best.pt"
CKPT.parent.mkdir(parents=True, exist_ok=True)
if not CKPT.is_file():
    run(sys.executable, "-m", "gdown", "1uCWam8EhAXt3z6QQAfhRfzHITUcml438", "-O", str(CKPT))
print("Setup complete:", CKPT)


In [ ]:
# Exact match supplied by the user: JaHbat vs FC-1, September 13, 2026.
# Downloads directly into Colab; no 28 GB transfer through the local computer.
import shutil
import gdown
FILE_ID = "1lkoCqWRF9CEvbzJp37LSU8asjE7jxTNp"
EXPECTED_BYTES = 28307958275
VIDEO_PATH = "/content/matches/JaHbat_vs_FC1_2026-09-13.mp4"
video_target = Path(VIDEO_PATH)
video_target.parent.mkdir(parents=True, exist_ok=True)
if not video_target.exists() or video_target.stat().st_size != EXPECTED_BYTES:
    assert shutil.disk_usage(video_target.parent).free > EXPECTED_BYTES + 2_000_000_000, "Not enough runtime disk space"
    partial = str(video_target) + ".partial"
    result = gdown.download(id=FILE_ID, output=partial, quiet=False, resume=True)
    assert result, "Drive requires authenticated access. Mount Drive and set VIDEO_PATH to the shared match before continuing."
    assert Path(partial).stat().st_size == EXPECTED_BYTES, "Download size mismatch"
    Path(partial).replace(video_target)
THRESHOLD = 0.5
BATCH_SIZE = 1
print("Match ready:", VIDEO_PATH)


In [ ]:
assert VIDEO_PATH, "Set VIDEO_PATH to the full match first"
video = Path(VIDEO_PATH)
assert video.is_file(), f"Video not found: {video}"
sys.path.insert(0, str(REPO / "scripts"))
import evaluate_gar as eg  # Applies the Linux ffmpeg compatibility fix.
import numpy as np
from collections import Counter
tr = eg.tr
duration = tr.probe_duration(video)
assert duration > 0, "Video has no measurable duration"
print(f"Processing entire supplied match file: {duration / 60:.1f} minutes")
out = REPO / "outputs/full_match_a100" / time.strftime("%Y%m%d_%H%M%S")
out.mkdir(parents=True)
last_print = [0.0]
def progress(fraction, message):
    now = time.monotonic()
    if now - last_print[0] >= 30 or fraction >= 1:
        print(message, flush=True)
        last_print[0] = now
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
model = tr.get_model("cuda")
scores = tr.compute_scores(video, model=model, start=0, duration=None,
                           batch_size=BATCH_SIZE, use_cache=False, progress=progress)
torch.cuda.synchronize()
elapsed = time.perf_counter() - started
assert np.isfinite(scores).all(), "Nonfinite model scores"
coverage = len(scores) / tr.SAMPLED_FPS
assert abs(coverage - duration) <= 2, f"Incomplete decode: {coverage} of {duration} seconds"
events = tr.extract_events(scores, threshold=THRESHOLD)
paths = tr.save_events(events, video, THRESHOLD, out / "events.json")
metrics = dict(gpu=GPU, revision=REVISION, video=video.name,
               duration_seconds=duration, processed_seconds=coverage,
               elapsed_seconds=elapsed, realtime_factor=duration / elapsed,
               peak_gpu_memory_gb=torch.cuda.max_memory_allocated() / 1e9,
               batch_size=BATCH_SIZE, threshold=THRESHOLD, cached=False,
               event_count=len(events), per_class=dict(Counter(e['label'] for e in events)))
(out / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print("Results:", out)
